In [ ]:
!pip install spacy

In [2]:
# Import Libraries
import pandas as pd
import spacy
from spacy.matcher import Matcher

In [ ]:
# load Spacy model
!python -m spacy download en_core_web_md
nlp = spacy.load("en_core_web_md")

In [4]:
# Create the dataset
data = {
    "ticket": [
        "My payment failed",
        "Payment was declined",
        "Money was deducted from my account",
        "My credit card payment is not working",
        "There is an issue with my billing",

        "Unable to login to my account",
        "I forgot my password",
        "The website is not working",
        "I cannot access my account",
        "The app keeps crashing",

        "I need a refund for my order",
        "How can I get my money back",
        "I want to return my order",
        "Please process my refund",
        "I received a damaged product and want a refund",

        "I want to know about your pricing",
        "What plans do you offer",
        "I am interested in your premium plan",
        "Can someone explain your subscription plans",
        "I want to purchase your service"
    ],

    "department": [
        "Billing", "Billing", "Billing", "Billing", "Billing",
        "Technical", "Technical", "Technical", "Technical", "Technical",
        "Refund", "Refund", "Refund", "Refund", "Refund",
        "Sales", "Sales", "Sales", "Sales", "Sales"
    ]
}

df = pd.DataFrame(data)

df.head()

,ticket,department
0,My payment failed,Billing
1,Payment was declined,Billing
2,Money was deducted from my account,Billing
3,My credit card payment is not working,Billing
4,There is an issue with my billing,Billing


In [5]:
# Check the departments
df["department"].value_counts()

department
Billing      5
Technical    5
Refund       5
Sales        5
Name: count, dtype: int64

In [6]:
# Basic spaCy NLP
text = df["ticket"][0]

doc = nlp(text)

for token in doc:
    print(token.text, token.pos_, token.lemma_)

My PRON my
payment NOUN payment
failed VERB fail


In [7]:
# Create Matcher
matcher = Matcher(nlp.vocab)

In [8]:
# Add department patterns
patterns = {
    "Billing": [
        [{"LOWER": "payment"}],
        [{"LOWER": "billing"}],
        [{"LOWER": "credit"}, {"LOWER": "card"}],
        [{"LOWER": "money"}, {"LOWER": "deducted"}],
        [{"LOWER": "charged"}],
        [{"LOWER": "transaction"}]
    ],

    "Technical": [
        [{"LOWER": "login"}],
        [{"LOWER": "password"}],
        [{"LOWER": "website"}],
        [{"LOWER": "app"}],
        [{"LOWER": "account"}],
        [{"LOWER": "access"}],
        [{"LOWER": "crashing"}]
    ],

    "Refund": [
        [{"LOWER": "refund"}],
        [{"LOWER": "return"}],
        [{"LOWER": "money"}, {"LOWER": "back"}],
        [{"LOWER": "damaged"}, {"LOWER": "product"}]
    ],

    "Sales": [
        [{"LOWER": "pricing"}],
        [{"LOWER": "plans"}],
        [{"LOWER": "premium"}],
        [{"LOWER": "subscription"}],
        [{"LOWER": "purchase"}],
        [{"LOWER": "offer"}]
    ]
}

In [9]:
# Add patterns to Matcher
for department, pattern_list in patterns.items():
    matcher.add(department, pattern_list)

In [10]:
# Test matcher
text = "My payment failed"

doc = nlp(text)

matches = matcher(doc)

for match_id, start, end in matches:
    print("Department:", nlp.vocab.strings[match_id])
    print("Matched word:", doc[start:end].text)

Department: Billing
Matched word: payment


In [11]:
# Prediction function
def predict_department(ticket):

    doc = nlp(ticket)

    matches = matcher(doc)

    if matches:
        match_id, start, end = matches[0]
        return nlp.vocab.strings[match_id]

    return "Unknown"

In [12]:
# Test one ticket
ticket = "My payment failed"

print("Ticket:", ticket)
print("Predicted Department:", predict_department(ticket))

Ticket: My payment failed
Predicted Department: Billing


In [13]:
# Test original dataset
for index, row in df.iterrows():

    print("Ticket:", row["ticket"])
    print("Actual:", row["department"])
    print("Predicted:", predict_department(row["ticket"]))
    print("-" * 40)

Ticket: My payment failed
Actual: Billing
Predicted: Billing
----------------------------------------
Ticket: Payment was declined
Actual: Billing
Predicted: Billing
----------------------------------------
Ticket: Money was deducted from my account
Actual: Billing
Predicted: Technical
----------------------------------------
Ticket: My credit card payment is not working
Actual: Billing
Predicted: Billing
----------------------------------------
Ticket: There is an issue with my billing
Actual: Billing
Predicted: Billing
----------------------------------------
Ticket: Unable to login to my account
Actual: Technical
Predicted: Technical
----------------------------------------
Ticket: I forgot my password
Actual: Technical
Predicted: Technical
----------------------------------------
Ticket: The website is not working
Actual: Technical
Predicted: Technical
----------------------------------------
Ticket: I cannot access my account
Actual: Technical
Predicted: Technical
----------------

In [14]:
# Test department wise
for department in ["Billing", "Technical", "Refund", "Sales"]:

    print("\n" + "=" * 40)
    print(department)
    print("=" * 40)

    team_tickets = df[df["department"] == department]

    for index, row in team_tickets.iterrows():

        print("Ticket:", row["ticket"])
        print("Actual:", row["department"])
        print("Predicted:", predict_department(row["ticket"]))
        print()


Billing
Ticket: My payment failed
Actual: Billing
Predicted: Billing

Ticket: Payment was declined
Actual: Billing
Predicted: Billing

Ticket: Money was deducted from my account
Actual: Billing
Predicted: Technical

Ticket: My credit card payment is not working
Actual: Billing
Predicted: Billing

Ticket: There is an issue with my billing
Actual: Billing
Predicted: Billing


Technical
Ticket: Unable to login to my account
Actual: Technical
Predicted: Technical

Ticket: I forgot my password
Actual: Technical
Predicted: Technical

Ticket: The website is not working
Actual: Technical
Predicted: Technical

Ticket: I cannot access my account
Actual: Technical
Predicted: Technical

Ticket: The app keeps crashing
Actual: Technical
Predicted: Technical


Refund
Ticket: I need a refund for my order
Actual: Refund
Predicted: Refund

Ticket: How can I get my money back
Actual: Refund
Predicted: Refund

Ticket: I want to return my order
Actual: Refund
Predicted: Refund

Ticket: Please process my r

In [15]:
# Test a new customer ticket (Billing)
ticket = "My payment was declined"

actual_department = "Billing"

predicted_department = predict_department(ticket)

print("Ticket:", ticket)
print("Actual Department:", actual_department)
print("Predicted Department:", predicted_department)

if actual_department == predicted_department:
    print("Result: Correct")
    print("Accuracy: 100%")
else:
    print("Result: Incorrect")
    print("Accuracy: 0%")

Ticket: My payment was declined
Actual Department: Billing
Predicted Department: Billing
Result: Correct
Accuracy: 100%


In [16]:
# Test a new customer ticket (Technical)
ticket = "Unable to login to my account"

actual_department = "Technical"

predicted_department = predict_department(ticket)

print("Ticket:", ticket)
print("Actual Department:", actual_department)
print("Predicted Department:", predicted_department)

if actual_department == predicted_department:
    print("Result: Correct")
    print("Accuracy: 100%")
else:
    print("Result: Incorrect")
    print("Accuracy: 0%")

Ticket: Unable to login to my account
Actual Department: Technical
Predicted Department: Technical
Result: Correct
Accuracy: 100%


In [17]:
# Test a new customer ticket (Refund)
ticket = "How can I get my money back"

actual_department = "Refund"

predicted_department = predict_department(ticket)

print("Ticket:", ticket)
print("Actual Department:", actual_department)
print("Predicted Department:", predicted_department)

if actual_department == predicted_department:
    print("Result: Correct")
    print("Accuracy: 100%")
else:
    print("Result: Incorrect")
    print("Accuracy: 0%")

Ticket: How can I get my money back
Actual Department: Refund
Predicted Department: Refund
Result: Correct
Accuracy: 100%


In [18]:
# Test a new customer ticket (Sales)
ticket = "I want to know about your pricing"

actual_department = "Sales"

predicted_department = predict_department(ticket)

print("Ticket:", ticket)
print("Actual Department:", actual_department)
print("Predicted Department:", predicted_department)

if actual_department == predicted_department:
    print("Result: Correct")
    print("Accuracy: 100%")
else:
    print("Result: Incorrect")
    print("Accuracy: 0%")

Ticket: I want to know about your pricing
Actual Department: Sales
Predicted Department: Sales
Result: Correct
Accuracy: 100%


In [19]:
# Basic NER Example
text = "I need a refund for order 45892"

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, ent.label_)

45892 CARDINAL


In [20]:
# Test multiple new tickets
new_tickets = [
    ("My payment is not working", "Billing"),
    ("I cannot login to my account", "Technical"),
    ("I want my money back", "Refund"),
    ("What is your premium plan?", "Sales")
]

correct = 0

for ticket, actual in new_tickets:

    predicted = predict_department(ticket)

    print("Ticket:", ticket)
    print("Actual:", actual)
    print("Predicted:", predicted)

    if actual == predicted:
        correct += 1
        print("Result: Correct")
    else:
        print("Result: Incorrect")

    print("-" * 40)

accuracy = correct / len(new_tickets)

print("Final Accuracy:", accuracy * 100, "%")

Ticket: My payment is not working
Actual: Billing
Predicted: Billing
Result: Correct
----------------------------------------
Ticket: I cannot login to my account
Actual: Technical
Predicted: Technical
Result: Correct
----------------------------------------
Ticket: I want my money back
Actual: Refund
Predicted: Refund
Result: Correct
----------------------------------------
Ticket: What is your premium plan?
Actual: Sales
Predicted: Sales
Result: Correct
----------------------------------------
Final Accuracy: 100.0 %


In [23]:
# Interactive Ticket Testing

while True:

    ticket = input("\nEnter customer ticket (type 'exit' to stop): ")

    if ticket.lower() == "exit":
        break

    doc = nlp(ticket)

    # Matcher
    matches = matcher(doc)

    if matches:
        match_id, start, end = matches[0]
        department = nlp.vocab.strings[match_id]
        matched_word = doc[start:end].text
    else:
        department = "Unknown"
        matched_word = "No match"

    # NER
    entities = []

    for ent in doc.ents:
        entities.append(f"{ent.text} → {ent.label_}")

    # Dataset accuracy
    df["predicted_department"] = df["ticket"].apply(predict_department)

    # Accuracy for this ticket
    if predicted.lower() == actual.lower():
        accuracy = 100
    else:
        accuracy = 0

    # Results
    print("=" * 40)
    print("Ticket:", ticket)
    print("Matcher:", matched_word)
    print("Predicted Department:", department)

    if entities:
        print("NER:", ", ".join(entities))
    else:
        print("NER: No entities found")

    print("Model Accuracy:", round(accuracy, 2), "%")

    print("=" * 40)


Enter customer ticket (type 'exit' to stop):  Payment was declined


Ticket: Payment was declined
Matcher: Payment
Predicted Department: Billing
NER: No entities found
Model Accuracy: 100 %



Enter customer ticket (type 'exit' to stop):  The app keeps crashing


Ticket: The app keeps crashing
Matcher: app
Predicted Department: Technical
NER: No entities found
Model Accuracy: 100 %



Enter customer ticket (type 'exit' to stop):  How can I get my money back


Ticket: How can I get my money back
Matcher: money back
Predicted Department: Refund
NER: No entities found
Model Accuracy: 100 %



Enter customer ticket (type 'exit' to stop):  What plans do you offer


Ticket: What plans do you offer
Matcher: plans
Predicted Department: Sales
NER: No entities found
Model Accuracy: 100 %



Enter customer ticket (type 'exit' to stop):  exit
